# 🌌 PolyMerge: Universal Architecture-Agnostic LLM Merger & Toolkit

Interactive Production Notebook for **Google Colab** and **Kaggle**.

### ⚡ Key Capabilities:
1. **Core Model Merging**: 12 Methods (`linear`, `slerp`, `nuslerp`, `karcher`, `nearswap`, `task_arithmetic`, `ties`, `dare_ties`, `della`, `omnimerge_v2`, `model_stock`, `sce`).
2. **Vision / Multimodal Auto-Filter**: Automatically isolates text backbone and drops vision layers (`visual.*`, `vision_tower.*`, `mm_projector.*`).
3. **Zero-Class Streaming**: Directly manipulates raw Safetensors and GGUF files with lowest RAM consumption and sequential cache eviction.
4. **MoE Upcycling**: Dense-to-MoE assembly (`Mixtral`, `Qwen-MoE`, `DeepSeek-MoE`, `Gemma 4 MoE`).
5. **Layer Slicing & Frankenmerging**: Arbitrary layer duplication and stacking (e.g. 32L -> 48L).
6. **Model Pruning**: Zero-retraining Dense Layer Pruning (Local Residual Healing) & MoE Expert Pruning (T18 Soft-Transfer).
7. **LoRA Toolkit & Tokenizer Surgery**: Truncated SVD extraction, permanent base baking, vocabulary alignment, and truncation.
8. **Sanity Audit & Direct HF Push**: Check NaN/Inf parameters, router condition numbers, and publish straight to Hugging Face Hub.

---  
## 1. 📦 Installation & Setup

In [ ]:
#@title 1. Install Dependencies & Clone PolyMerge
!pip install -q pyyaml rich huggingface-hub safetensors hf-transfer
!rm -rf polymerge
!git clone https://github.com/ehristoforu/polymerge.git
%cd polymerge
!pip install -q -e .
print("✓ PolyMerge successfully installed!")

---  
## 2. ⚙️ Hardware Platform Configuration

In [ ]:
#@title 2. Select Hardware Platform & Optimization Tuning
platform = "gpu" #@param ["cpu", "cpu+highram", "gpu", "tpu"]
chunk_size_mb = 2000 #@param {type:"integer"}

import os
import torch

os.environ["POLYMERGE_DEVICE"] = "auto"
chunk_bytes = chunk_size_mb * 1_000_000

if platform == "cpu":
    os.environ["POLYMERGE_DEVICE"] = "cpu"
    print("⚙️ Platform: CPU Standard. Chunking set to:", chunk_size_mb, "MB")
elif platform == "cpu+highram":
    os.environ["POLYMERGE_DEVICE"] = "cpu"
    chunk_bytes = 4_000_000_000
    print("⚙️ Platform: CPU High-RAM. Shard chunk size expanded to 4GB.")
elif platform == "gpu":
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        os.environ["POLYMERGE_DEVICE"] = "cuda"
        print(f"⚙️ Platform: GPU ({gpu_name}, {vram_gb:.1f} GB VRAM). Hardware acceleration ACTIVE.")
    else:
        print("⚠️ GPU not detected! Falling back to CPU. (Change Runtime type to T4 GPU)")
        os.environ["POLYMERGE_DEVICE"] = "cpu"
elif platform == "tpu":
    try:
        import torch_xla.core.xla_model as xm
        os.environ["POLYMERGE_DEVICE"] = "tpu"
        print("⚙️ Platform: Cloud TPU detected! Hardware routing via PyTorch/XLA.")
    except ImportError:
        print("⚠️ PyTorch/XLA not found. Installing torch_xla for TPU support...")
        !pip install -q torch_xla
        os.environ["POLYMERGE_DEVICE"] = "tpu"
        print("⚙️ TPU support installed.")

---  
## 3. 🔑 Secrets & Hugging Face High-Speed Transfer

In [ ]:
#@title 3. Hugging Face Login (Secrets & Fast Transfer)
import os
from huggingface_hub import login

# Enable high-speed Rust-based multi-threaded transfer
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print("✓ Found HF_TOKEN in Colab Secrets.")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("✓ Successfully authenticated with Hugging Face!")
else:
    print("ℹ️ HF_TOKEN not set in Colab Secrets. You can enter it manually below if accessing gated/private models:")
    # login() # Uncomment to login interactively

---  
## 4. 📝 Multi-Line YAML Config Definition

In [ ]:
#@title 4. Define Merge YAML Configuration
yaml_config = """
merge_method: dare_ties
base_model: meta-llama/Llama-3.2-1B
models:
  - model: meta-llama/Llama-3.2-1B-Instruct
    parameters:
      weight: 0.6
      density: 0.5
  - model: meta-llama/Llama-3.2-1B
    parameters:
      weight: 0.4
      density: 0.5
parameters:
  normalize: true
dtype: bfloat16
ignore_vision: true
"""

# Save configuration to disk
config_path = "merge_config.yaml"
with open(config_path, "w", encoding="utf-8") as f:
    f.write(yaml_config.strip())

print(f"✓ Config written to {config_path}:")
print("--------------------------------------------------")
print(yaml_config.strip())
print("--------------------------------------------------")

---  
## 5. 🔍 Resource Audit & Environment Validation

In [ ]:
#@title 5. Audit Hardware Resources & Environment Health
import psutil
import torch
from polymerge.utils import human_bytes

ram = psutil.virtual_memory()
disk = psutil.disk_usage('/')

print("=================== ENVIRONMENT AUDIT ===================")
print(f"• System RAM Available : {human_bytes(ram.available)} / {human_bytes(ram.total)}")
print(f"• Disk Space Free      : {human_bytes(disk.free)} / {human_bytes(disk.total)}")
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory
    print(f"• GPU Accelerator      : {torch.cuda.get_device_name(0)} ({human_bytes(vram)} VRAM)")
else:
    print("• Compute Device       : CPU Mode")
print("• Selected Target Device:", os.environ.get("POLYMERGE_DEVICE", "auto"))
print("=========================================================")

---  
## 6. 🚀 Launch PolyMerge Merge Execution

In [ ]:
#@title 6. Execute Model Merge
output_directory = "./merged_model" #@param {type:"string"}

!polymerge merge_config.yaml -o {output_directory} --chunk-size {chunk_bytes}

# Optional in-flight sanity audit
print("\nRunning in-flight model sanity audit...")
!polymerge --audit --base {output_directory}

---  
## 7. 📤 Direct Publishing to Hugging Face Hub

In [ ]:
#@title 7. Publish Result to Hugging Face Hub
hub_repo_id = "username/my-merged-model" #@param {type:"string"}
make_private = False #@param {type:"boolean"}
commit_msg = "Upload merged model via PolyMerge Colab" #@param {type:"string"}

import os
from pathlib import Path
from polymerge.publisher import PublisherTool

# Automatically check and use the output directory from the previous cell
target_dir = output_directory if 'output_directory' in locals() and Path(output_directory).exists() else "./merged_model"

if hub_repo_id == "username/my-merged-model" or not hub_repo_id.strip():
    print("⚠️ Please provide a valid Hugging Face repo ID (e.g. 'my-username/my-awesome-merge')!")
else:
    token_to_use = hf_token if 'hf_token' in locals() and hf_token else None
    print(f"Publishing model from '{target_dir}' to https://huggingface.co/{hub_repo_id}...")
    PublisherTool.push_to_hub(
        model_dir=target_dir,
        repo_id=hub_repo_id.strip(),
        private=make_private,
        token=token_to_use,
        commit_message=commit_msg,
    )
    print(f"\n✓ Model successfully published to: https://huggingface.co/{hub_repo_id}")